# MaaS Advanced Features

This notebook explores MaaS platform features beyond basic inference:

1. **Subscriptions & Rate Limits** — Configure token rate limit policies per group
2. **Multi-Tier Demo** — Free vs Premium subscription comparison
3. **API Key Management** — Create and list API keys
4. **Monitoring & Observability** — Query Prometheus metrics, check usage

> Test resources created here are cleaned up in `2_maas_policy_test.ipynb`.

**Prerequisites:**
- MaaS enabled with models registered (`../2_maas/2_enable_maas.ipynb` completed)
- Model serving tested (`../2_maas/3_test_model_serving.ipynb`)
- Cluster-admin or equivalent permissions for subscription/policy configuration

In [ ]:
import subprocess, json, time, os
import urllib.request, urllib.parse, ssl
from dotenv import load_dotenv

load_dotenv("../.env")

CLUSTER_DOMAIN = os.getenv("CLUSTER_DOMAIN")
if not CLUSTER_DOMAIN:
    r = subprocess.run(["oc", "get", "ingresses.config.openshift.io", "cluster",
                        "-o", "jsonpath={.spec.domain}"], capture_output=True, text=True)
    CLUSTER_DOMAIN = r.stdout.strip()

MODEL_NAMESPACE = os.getenv("MODEL_NAMESPACE", "demo")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen36-27b")

MAAS_HOST = f"https://maas-api.{CLUSTER_DOMAIN}"
INFERENCE_GW = os.getenv("MODEL_ENDPOINT", f"https://maas-api.{CLUSTER_DOMAIN}/{MODEL_NAMESPACE}/{MODEL_NAME}")

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print(f"MaaS API:       {MAAS_HOST}/maas-api/v1")
print(f"Inference GW:   {INFERENCE_GW}")
print(f"Model:          {MODEL_NAMESPACE}/{MODEL_NAME}")
print(f"OCP User:       {subprocess.run(['oc', 'whoami'], capture_output=True, text=True).stdout.strip()}")

---
## 1. Subscriptions & Token Rate Limits

MaaS uses **MaaSSubscription** CRDs to define per-group token rate limits.

```
MaaSAuthPolicy   → Who can access which models
MaaSSubscription → How much quota they get (tokens per window)
```

Ref: [Quota and Access Configuration](https://opendatahub-io.github.io/models-as-a-service/latest/configuration-and-management/quota-and-access-configuration/)

### 1.1 View Existing Subscriptions and Policies

In [ ]:
%%bash
source ../.env 2>/dev/null || true
MODEL_NS=${MODEL_NAMESPACE:-demo}

echo "=== MaaSModelRefs (registered models) ==="
oc get maasmodelref -n ${MODEL_NS} 2>/dev/null || echo "No MaaSModelRef resources found"

echo ""
echo "=== MaaSAuthPolicies (access control) ==="
oc get maasauthpolicy -n models-as-a-service 2>/dev/null || echo "No MaaSAuthPolicy resources found"

echo ""
echo "=== MaaSSubscriptions (rate limits) ==="
oc get maassubscription -n models-as-a-service 2>/dev/null || echo "No MaaSSubscription resources found"

echo ""
echo "=== Generated Kuadrant Policies ==="
echo "AuthPolicies:"
oc get authpolicy -n ${MODEL_NS} --no-headers 2>/dev/null || echo "  None"
echo "TokenRateLimitPolicies:"
oc get tokenratelimitpolicy -n ${MODEL_NS} --no-headers 2>/dev/null || echo "  None"

### 1.2 View Subscription Details

In [ ]:
%%bash
echo "=== MaaSSubscription Details ==="
for sub in $(oc get maassubscription -n models-as-a-service -o jsonpath='{.items[*].metadata.name}' 2>/dev/null); do
    echo ""
    echo "--- ${sub} ---"
    oc get maassubscription ${sub} -n models-as-a-service -o json 2>/dev/null | \
        python3 -c "
import sys, json
data = json.load(sys.stdin)
spec = data.get('spec', {})
print(f'  Priority: {spec.get(\"priority\", \"N/A\")}')
print(f'  Owner Groups: {[g[\"name\"] for g in spec.get(\"owner\", {}).get(\"groups\", [])]}')
for ref in spec.get('modelRefs', []):
    limits = ref.get('tokenRateLimits', [])
    for lim in limits:
        print(f'  Model: {ref[\"namespace\"]}/{ref[\"name\"]}  →  {lim[\"limit\"]} tokens / {lim[\"window\"]}')
"
done

---
## 2. Multi-Tier Demo: Free vs Premium

This demonstrates why subscription-based rate limiting matters:
- **Free tier**: 500 tokens/min (quickly hits 429)
- **Premium tier**: 50000 tokens/min (comfortable headroom)

We create two groups, two subscriptions, and show the difference.

In [ ]:
%%bash
source ../.env 2>/dev/null || true
MODEL_NS=${MODEL_NAMESPACE:-demo}
MODEL_REF=$(oc get maasmodelref -n ${MODEL_NS} -o jsonpath='{.items[0].metadata.name}' 2>/dev/null)

if [ -z "$MODEL_REF" ]; then
    echo "ERROR: No MaaSModelRef found in namespace '${MODEL_NS}'."
    echo "Run ../2_maas/2_enable_maas.ipynb first."
    exit 1
fi

echo "Using model ref: ${MODEL_NS}/${MODEL_REF}"
echo ""

OCP_USER=$(oc whoami)
echo "Setting up multi-tier subscriptions for user: ${OCP_USER}"

echo ""
echo "=== 2.1 Create Free access policy ==="
oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSAuthPolicy
metadata:
  name: lab-free-access
  namespace: models-as-a-service
spec:
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
  subjects:
    groups: []
    users:
      - "${OCP_USER}"
EOF

echo ""
echo "=== 2.2 Create Premium access policy ==="
oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSAuthPolicy
metadata:
  name: lab-premium-access
  namespace: models-as-a-service
spec:
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
  subjects:
    groups: []
    users:
      - "${OCP_USER}"
EOF

echo ""
echo "=== 2.3 Create Free subscription (500 tokens/min, priority 5) ==="
oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSSubscription
metadata:
  name: lab-free-subscription
  namespace: models-as-a-service
spec:
  owner:
    groups: []
    users:
      - "${OCP_USER}"
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
      tokenRateLimits:
        - limit: 500
          window: 1m
  priority: 5
EOF

echo ""
echo "=== 2.4 Create Premium subscription (50000 tokens/min, priority 20) ==="
oc apply -f - <<EOF
apiVersion: maas.opendatahub.io/v1alpha1
kind: MaaSSubscription
metadata:
  name: lab-premium-subscription
  namespace: models-as-a-service
spec:
  owner:
    groups: []
    users:
      - "${OCP_USER}"
  modelRefs:
    - name: ${MODEL_REF}
      namespace: ${MODEL_NS}
      tokenRateLimits:
        - limit: 50000
          window: 1m
  priority: 20
EOF

echo ""
echo "Waiting for policies to reconcile (30s)..."
sleep 30
echo "Done. Verify:"
oc get maassubscription -n models-as-a-service --no-headers

### 2.6 Create API Keys for Each Tier

Each API key is bound to one subscription at creation time.

In [ ]:
def create_api_key(name, subscription=None):
    data = {"name": name, "expiresIn": "1h"}
    if subscription:
        data["subscription"] = subscription
    body = json.dumps(data).encode()
    req = urllib.request.Request(
        f"{MAAS_HOST}/maas-api/v1/api-keys",
        data=body,
        headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(req, context=ctx, timeout=15) as resp:
            result = json.loads(resp.read())
            return result.get("key", ""), result.get("subscription", "")
    except urllib.error.HTTPError as e:
        body = e.read().decode() if e.fp else ""
        print(f"  ERROR creating key '{name}': HTTP {e.code} — {body[:200]}")
        return "", ""
    except Exception as e:
        print(f"  ERROR: {e}")
        return "", ""

print("Creating API keys for each tier...")
print("")

FREE_KEY, free_sub = create_api_key("free-tier-test", "lab-free-subscription")
print(f"  Free key:    {FREE_KEY[:20]}...  (subscription: {free_sub})")

PREMIUM_KEY, premium_sub = create_api_key("premium-tier-test", "lab-premium-subscription")
print(f"  Premium key: {PREMIUM_KEY[:20]}...  (subscription: {premium_sub})")

if not FREE_KEY or not PREMIUM_KEY:
    print("\nWARNING: Could not create API keys. Check MaaS API status.")
    print("Hint: oc get pods -l app.kubernetes.io/name=maas-api -A")

### 2.7 Test Free Tier (expect 429 quickly)

In [ ]:
if FREE_KEY:
    print("Free Tier Test (500 tokens/min limit):")
    print("=" * 50)
    
    test_body = json.dumps({
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Write a haiku about coding."}],
        "max_tokens": 100
    }).encode()
    
    for i in range(8):
        req = urllib.request.Request(
            f"{INFERENCE_GW}/v1/chat/completions",
            data=test_body,
            headers={"Authorization": f"Bearer {FREE_KEY}", "Content-Type": "application/json"},
            method="POST"
        )
        try:
            with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
                print(f"  Request {i+1}: HTTP {resp.status}")
        except urllib.error.HTTPError as e:
            if e.code == 429:
                print(f"  Request {i+1}: HTTP 429 ← Rate limit hit!")
            else:
                print(f"  Request {i+1}: HTTP {e.code}")
        except Exception as e:
            print(f"  Request {i+1}: Error — {e}")
else:
    print("Skipped — no Free key available")

### 2.8 Test Premium Tier (should pass comfortably)

In [ ]:
if PREMIUM_KEY:
    print("Premium Tier Test (50000 tokens/min limit):")
    print("=" * 50)
    
    test_body = json.dumps({
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": "Write a haiku about coding."}],
        "max_tokens": 100
    }).encode()
    
    for i in range(5):
        req = urllib.request.Request(
            f"{INFERENCE_GW}/v1/chat/completions",
            data=test_body,
            headers={"Authorization": f"Bearer {PREMIUM_KEY}", "Content-Type": "application/json"},
            method="POST"
        )
        try:
            with urllib.request.urlopen(req, context=ctx, timeout=30) as resp:
                print(f"  Request {i+1}: HTTP {resp.status}")
        except urllib.error.HTTPError as e:
            print(f"  Request {i+1}: HTTP {e.code}")
        except Exception as e:
            print(f"  Request {i+1}: Error — {e}")
    
    print("\n  All passed — Premium tier has sufficient quota.")
else:
    print("Skipped — no Premium key available")

---
## 3. API Key Management

MaaS provides full lifecycle management: create, list, inspect, and revoke.

Ref: [API Key Management](https://opendatahub-io.github.io/models-as-a-service/latest/user-guide/api-key-management/)

### 3.1 List Active API Keys

In [ ]:
search_data = json.dumps({"status": "active", "limit": 20, "includeEphemeral": True}).encode()
req = urllib.request.Request(
    f"{MAAS_HOST}/maas-api/v1/api-keys/search",
    data=search_data,
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"},
    method="POST"
)

try:
    with urllib.request.urlopen(req, context=ctx, timeout=15) as resp:
        keys_list = json.loads(resp.read())

    items = keys_list if isinstance(keys_list, list) else keys_list.get("items", keys_list.get("data", []))
    print("Active API Keys")
    print("=" * 80)
    print(f"{'Name':<22} {'ID':<38} {'Subscription':<25} {'Expires'}")
    print("-" * 80)
    for key in items:
        name = key.get("name", "(ephemeral)")
        print(f"{name:<22} {key.get('id', ''):<38} {key.get('subscription', ''):<25} {key.get('expiresAt', '')}")
except urllib.error.HTTPError as e:
    print(f"ERROR listing keys: HTTP {e.code}")
    print(f"  Response: {e.read().decode()[:200]}")
except Exception as e:
    print(f"ERROR: {e}")

---
## 4. Monitoring & Observability

MaaS exposes metrics through Prometheus. Key metrics:

| Metric | Source | What It Measures |
|--------|--------|------------------|
| `authorized_calls` | Limitador | Successful requests (within rate limit) |
| `authorized_hits` | Limitador | Token usage (authorized tokens consumed) |
| `limited_calls` | Limitador | Rejected requests (rate limit exceeded) |
| `auth_server_evaluations_total` | Authorino | Auth evaluation count |

### 4.1 Check Observability Prerequisites

In [ ]:
%%bash
echo "=== User Workload Monitoring ==="
UWM_PODS=$(oc get pods -n openshift-user-workload-monitoring --no-headers 2>/dev/null | wc -l)
if [ "$UWM_PODS" -gt 0 ]; then
    echo "User Workload Monitoring active ($UWM_PODS pods)"
else
    echo "User Workload Monitoring not active"
    echo "  Enable: oc apply -f - <<< '{\"apiVersion\":\"v1\",\"kind\":\"ConfigMap\",\"metadata\":{\"name\":\"cluster-monitoring-config\",\"namespace\":\"openshift-monitoring\"},\"data\":{\"config.yaml\":\"enableUserWorkload: true\"}}'"
fi

echo ""
echo "=== Limitador Pods ==="
oc get pods -n kuadrant-system -l app=limitador --no-headers 2>/dev/null || echo "  Not found"

echo ""
echo "=== Component Health ==="
echo "Gateway:"
oc get gateway -n openshift-ingress maas-default-gateway -o jsonpath='  Programmed={.status.conditions[?(@.type=="Programmed")].status}' 2>/dev/null && echo ""
echo "Authorino:"
oc get pods -n kuadrant-system -l app=authorino --no-headers 2>/dev/null | head -1 || echo "  Not found"
echo "MaaS API:"
APP_NS=$(oc get ns redhat-ods-applications --no-headers 2>/dev/null && echo redhat-ods-applications || echo opendatahub)
oc get pods -n ${APP_NS} -l app.kubernetes.io/name=maas-api --no-headers 2>/dev/null | head -1 || echo "  Not found"

### 4.2 Query Prometheus Metrics

In [ ]:
THANOS_HOST = f"https://thanos-querier-openshift-monitoring.{CLUSTER_DOMAIN}"

queries = {
    "Authorized Calls (total)": "sum(authorized_calls)",
    "Authorized Tokens (total)": "sum(authorized_hits)",
    "Rate-Limited Calls (total)": "sum(limited_calls)",
    "Limitador Up": "limitador_up",
}

print("MaaS Prometheus Metrics")
print("=" * 60)

for label, query in queries.items():
    try:
        url = f"{THANOS_HOST}/api/v1/query?query={urllib.parse.quote(query)}"
        req = urllib.request.Request(url, headers={"Authorization": f"Bearer {OC_TOKEN}"})
        with urllib.request.urlopen(req, context=ctx, timeout=10) as resp:
            data = json.loads(resp.read())
            results = data.get("data", {}).get("result", [])
            if results:
                value = results[0].get("value", ["", "N/A"])[1]
                print(f"  {label:<35} {value}")
            else:
                print(f"  {label:<35} (no data)")
    except Exception as e:
        print(f"  {label:<35} Error: {str(e)[:40]}")

---
## Summary

| Feature | What We Tested |
|---------|----------------|
| **Subscriptions** | Created Free (500 tok/min) and Premium (50000 tok/min) subscriptions |
| **Multi-Tier** | Free hits 429 quickly while Premium passes comfortably |
| **API Key Lifecycle** | Created tier-bound keys, listed active keys |
| **Monitoring** | Queried Prometheus for authorized/limited calls and token usage |
| **Health Check** | Verified Gateway, Authorino, Limitador, MaaS API status |

### Key MaaS CRDs

| CRD | Namespace | Purpose |
|-----|-----------|--------|
| `MaaSModelRef` | Model namespace | Registers a model for MaaS |
| `MaaSAuthPolicy` | `models-as-a-service` | Grants group/user access to models |
| `MaaSSubscription` | `models-as-a-service` | Defines token rate limits per group |

### Reference

- [Quota & Access Configuration](https://opendatahub-io.github.io/models-as-a-service/latest/configuration-and-management/quota-and-access-configuration/)
- [API Key Management](https://opendatahub-io.github.io/models-as-a-service/latest/user-guide/api-key-management/)
- [Observability](https://opendatahub-io.github.io/models-as-a-service/latest/advanced-administration/observability/)